# S3.16 — Multiple Aggregations in One Pass
**Date completed:** October 2026  
**Status:** Completed  
**Dataset:** samples.tpch.orders — 7.5M rows
**Key concept:** One groupBy, many measures, one shuffle

In [0]:
# ============================================================
# S3.16 — Cell 2
# Conditional Aggregation, Multiple Measures and Pivot
#
# Goal:
# 1. Use conditional aggregation with sum(when(...)).
# 2. Calculate many measures at the same grain.
# 3. Understand NULL behaviour in conditional aggregation.
# 4. Build pivot-style summaries using conditional aggregation.
# 5. Use the actual pivot() API.
# 6. Build aggregation expressions programmatically.
# 7. Reconcile grouped counts and revenue to source totals.
# 8. Compare combined vs separate aggregation plans.
# 9. Understand performance implications without assuming
#    an exact number of shuffles.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Important:
# When multiple measures share the same grouping keys,
# calculating them together often allows Spark to share
# grouping work.
#
# However:
# One groupBy does NOT guarantee exactly one physical shuffle.
# Always inspect the physical plan / Query Profile when needed.
# ============================================================

import pyspark.sql.functions as F


print("=== S3.16: CONDITIONAL AGGREGATION AND PIVOT ===")
print()

# ------------------------------------------------------------
# Step 1 — Load source table
# ------------------------------------------------------------

df_orders = spark.table("samples.tpch.orders")

print("Orders DataFrame loaded.")
print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Conditional Aggregation per Customer
#
# Target grain:
#
# ONE ROW PER CUSTOMER
#
# We calculate:
#
# total_orders
# total_revenue
# fulfilled_orders
# open_orders
# fulfilled_revenue
# open_revenue
#
#
# Conditional aggregation pattern:
#
# SUM(
#     WHEN condition
#     THEN value
#     ELSE 0
# )
#
#
# Why ELSE 0?
#
# If we write:
#
# sum(when(condition, value))
#
# and a group has no matching rows,
# the aggregate can return NULL.
#
# Using otherwise(0) makes a no-match metric equal to zero,
# which is appropriate here because:
#
# no matching orders
#     =
# zero matching revenue/orders.
# ------------------------------------------------------------

print(
    "=== STEP 2: CONDITIONAL AGGREGATION PER CUSTOMER ==="
)


df_cust = (

    df_orders

    .groupBy("o_custkey")

    .agg(

        # Total orders for the customer
        F.count("*").alias("total_orders"),

        # Total revenue for the customer
        F.sum("o_totalprice").alias("total_revenue"),

        # Fulfilled-order revenue
        F.sum(
            F.when(F.col("o_orderstatus") == "F",F.col("o_totalprice"))
            .otherwise(F.lit(0))).alias("fulfilled_revenue"),

        # Open-order revenue
        F.sum(
            F.when(F.col("o_orderstatus") == "O",F.col("o_totalprice"))
            .otherwise(
                F.lit(0))).alias("open_revenue"),

        # Fulfilled-order count
        F.sum(
            F.when(F.col("o_orderstatus") == "F",1)
            .otherwise(0)).alias("fulfilled_orders"),

        # Open-order count
        F.sum(F.when(F.col("o_orderstatus") == "O",1)
            .otherwise(0)).alias("open_orders")))


print("\nTop customers by total revenue:")

(
    df_cust

    .orderBy(
        F.col("total_revenue").desc())

    .show(5,truncate=False))

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Important Conditional Aggregation Behaviour
#
# Another valid pattern is:
#
# count(
#     when(condition, 1)
# )
#
# because count(expression) ignores NULL values.
#
# Example:
#
# count(
#     when(status == "F", 1)
# )
#
# However, for conditional counts we use:
#
# sum(when(..., 1).otherwise(0))
#
# here because the TRUE/FALSE counting logic is very explicit.
# ------------------------------------------------------------

print(
    "=== STEP 3: CONDITIONAL COUNT PATTERNS ==="
)


print("""
Two common conditional-count patterns:

1.

count(
    when(condition, 1)
)

Why it works:
Non-matching rows return NULL,
and count(expression) ignores NULL.


2.

sum(
    when(condition, 1)
    .otherwise(0)
)

Why it works:
TRUE  → 1
FALSE → 0

Both can be valid.

For this lesson we use the second form because the
conditional-count logic is explicit.
""")

In [0]:
# ------------------------------------------------------------
# Step 4 — Many Measures at One Grain
#
# Target grain:
#
# ONE ROW PER ORDER STATUS
#
# Related measures using the same grouping key can be
# calculated together in one agg().
#
# This lets Spark optimise shared grouping work.
#
# Do NOT interpret this as:
#
# "one agg always equals exactly one shuffle"
#
# Physical execution is determined by Spark.
# ------------------------------------------------------------

print(
    "=== STEP 4: MANY MEASURES AT STATUS GRAIN ==="
)


df_status = (

    df_orders

    .groupBy("o_orderstatus")

    .agg(

        F.count("*").alias("order_count"),

        F.sum("o_totalprice").alias("total_revenue"),

        F.avg("o_totalprice").cast("decimal(18,2)").alias("avg_order_value"),

        F.min("o_totalprice").alias("min_order_value"),

        F.max("o_totalprice").alias("max_order_value"),

        F.min("o_orderdate").alias("first_order_date"),

        F.max("o_orderdate").alias("last_order_date"),

        F.countDistinct("o_custkey").alias("distinct_customers"))

    .orderBy(F.col("order_count").desc()))

df_status.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Conditional Pivot-Style Aggregation
#
# Requirement:
#
# One row per YEAR.
#
# Revenue status categories should become separate columns:
#
# revenue_F
# revenue_O
# revenue_P
#
# Conditional aggregation gives us explicit control over
# each output measure.
# ------------------------------------------------------------

print(
    "=== STEP 5: CONDITIONAL PIVOT-STYLE SUMMARY ==="
)


df_pivot_style = (

    df_orders

    .groupBy(
        F.year("o_orderdate").alias("order_year"))

    .agg(

        F.sum(F.when(F.col("o_orderstatus") == "F",F.col("o_totalprice"))
            .otherwise(F.lit(0))).alias("revenue_F"),

        F.sum(F.when(F.col("o_orderstatus") == "O",F.col("o_totalprice"))
            .otherwise(F.lit(0))).alias("revenue_O"),

        F.sum(F.when(F.col("o_orderstatus") == "P",F.col("o_totalprice"))
            .otherwise(F.lit(0))).alias("revenue_P"),

        F.sum("o_totalprice").alias("revenue_total"))

    .orderBy("order_year"))


df_pivot_style.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — Actual pivot() API
#
# Spark also provides pivot().
#
# Target grain:
#
# ONE ROW PER YEAR
#
# Pivot column:
#
# o_orderstatus
#
# Values:
#
# F
# O
# P
#
#
# IMPORTANT:
#
# We explicitly provide the known status values.
#
# This:
#
# 1. Controls which columns are produced.
# 2. Avoids relying on Spark to dynamically discover
#    all distinct pivot values.
#
# High-cardinality pivot columns should be avoided because
# they can create extremely wide output schemas.
# ------------------------------------------------------------

print(
    "=== STEP 6: ACTUAL pivot() ==="
)


statuses = [
    "F",
    "O",
    "P"
]


df_pivot = (

    df_orders

    .withColumn(
        "order_year",
        F.year("o_orderdate"))

    .groupBy("order_year")

    .pivot("o_orderstatus",statuses).agg(F.sum("o_totalprice"))

    .select(

        "order_year",

        F.col("F").alias("revenue_F"),

        F.col("O").alias("revenue_O"),

        F.col("P").alias("revenue_P"))

    .orderBy("order_year"))

print("\nActual pivot() result:")

df_pivot.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Conditional Aggregation vs pivot()
# ------------------------------------------------------------

print(
    "=== STEP 7: CONDITIONAL AGGREGATION vs pivot() ==="
)


print("""
Conditional aggregation:

groupBy(...)
    ↓
agg(
    sum(when(...)),
    sum(when(...))
)

Advantages:
- Very explicit.
- Flexible conditions.
- Easy to combine different business rules.
- Output column names are directly controlled.


pivot():

groupBy(...)
    ↓
pivot(category)
    ↓
agg(...)

Advantages:
- Concise when category values naturally become columns.
- Convenient for matrix/report-style summaries.


Important:

pivot() can become expensive or create a very wide DataFrame
when the pivot column has many distinct values.

For stable known domains, explicitly providing pivot values
is usually clearer:

pivot(
    "status",
    ["F", "O", "P"]
)
""")

In [0]:
# ------------------------------------------------------------
# Step 8 — Build Aggregations Programmatically
#
# Sometimes many similar measures must be generated.
#
# Instead of manually writing:
#
# revenue_F
# revenue_O
# revenue_P
#
# we can build Column expressions in Python.
#
# This does NOT execute Spark yet.
#
# It simply creates a Python list containing Spark
# Column expressions.
# ------------------------------------------------------------

print(
    "=== STEP 8: PROGRAMMATIC AGGREGATIONS ==="
)


agg_exprs = [

    F.sum(

        F.when(F.col("o_orderstatus") == status,F.col("o_totalprice"))

        .otherwise(F.lit(0))).alias(f"revenue_{status}")

    for status in statuses
    ]

# Add total revenue to the expression list.

agg_exprs.append(

    F.sum(
        "o_totalprice"
    ).alias(
        "revenue_total"
    )
)


df_dynamic = (

    df_orders

    .groupBy(
        F.year("o_orderdate").alias("order_year"))

    .agg(*agg_exprs)

    .orderBy("order_year"))


print("\nSame business measures, generated programmatically:")

df_dynamic.show(5,truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 9 — What *agg_exprs Means
#
# agg_exprs is a Python list:
#
# [
#     Column<revenue_F>,
#     Column<revenue_O>,
#     Column<revenue_P>,
#     Column<revenue_total>
# ]
#
# agg(*agg_exprs)
#
# uses Python argument unpacking.
#
# Conceptually:
#
# agg(
#     expression_1,
#     expression_2,
#     expression_3,
#     expression_4
# )
# ------------------------------------------------------------

print(
    "=== STEP 9: ARGUMENT UNPACKING ==="
)


print(
    f"Number of aggregation expressions: "
    f"{len(agg_exprs)}"
)

print("""
*agg_exprs

means:

Pass each item in the Python list as a separate
argument to agg().

This is a Python feature, not a special Spark operator.
""")

In [0]:
# ------------------------------------------------------------
# Step 10 — Count Reconciliation
#
# Every source order belongs to exactly one order status.
#
# Therefore:
#
# SUM(status-level order counts)
#
# should equal:
#
# source row count.
# ------------------------------------------------------------

print(
    "=== STEP 10: COUNT RECONCILIATION ==="
)


source_count = df_orders.count()


grouped_count = (

    df_status

    .agg(
        F.sum("order_count").alias("reconciled_count"))

    .first()["reconciled_count"])


print(
    f"Source rows          : {source_count:,}"
)

print(
    f"Grouped order counts : {grouped_count:,}"
)

assert source_count == grouped_count, ("Count reconciliation failed")

print("Count reconciliation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Conditional Count Reconciliation
#
# Validate the fulfilled-order conditional measure.
#
# Direct source count:
#
# status = F
#
# should equal:
#
# SUM(fulfilled_orders)
#
# across customer groups.
# ------------------------------------------------------------

print(
    "=== STEP 11: CONDITIONAL COUNT VALIDATION ==="
)


source_fulfilled_count = (

    df_orders

    .filter(
        F.col(
            "o_orderstatus"
        ) == "F"
    )

    .count()
)


aggregated_fulfilled_count = (

    df_cust

    .agg(
        F.sum("fulfilled_orders").alias("fulfilled_orders"))

    .first()["fulfilled_orders"])

print(f"Source fulfilled rows     : "f"{source_fulfilled_count:,}")

print(
    f"Aggregated fulfilled rows : "
    f"{aggregated_fulfilled_count:,}"
)


assert (
    source_fulfilled_count
    ==
    aggregated_fulfilled_count
), "Conditional count reconciliation failed"

print("Conditional count reconciliation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 12 — Revenue Reconciliation
#
# Global revenue:
#
# SUM(o_totalprice)
#
# should equal:
#
# SUM(revenue_total)
#
# across yearly groups.
# ------------------------------------------------------------

print(
    "=== STEP 12: REVENUE RECONCILIATION ==="
)


global_revenue = (

    df_orders

    .agg(
        F.sum("o_totalprice").alias("total_revenue"))

    .first()["total_revenue"])

yearly_revenue = (

    df_dynamic

    .agg(
        F.sum("revenue_total").alias("total_revenue"))

    .first()["total_revenue"])

print(f"Global revenue : {global_revenue}")

print(f"Yearly revenue : {yearly_revenue}")


assert global_revenue == yearly_revenue, ("Revenue reconciliation failed")

print("Revenue reconciliation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 13 — Combined vs Separate groupBy Plans
#
# We compare:
#
# A. One grouped aggregation containing several measures.
#
# B. Separate grouped aggregations for individual measures.
#
# IMPORTANT:
#
# explain() shows plan structure.
#
# It does NOT prove actual runtime performance.
#
# If the separate DataFrames are independently actioned,
# each normally executes its own query plan unless results
# are reused through caching/materialisation or another
# optimisation strategy.
#
# Use Databricks Query Profile when actual runtime evidence
# is required.
# ------------------------------------------------------------

print(
    "=== STEP 13: PLAN COMPARISON ==="
)


df_combined = (

    df_orders

    .groupBy(
        "o_orderstatus"
    )

    .agg(

        F.count("*").alias(
            "order_count"
        ),

        F.sum(
            "o_totalprice"
        ).alias(
            "total_revenue"
        )
    )
)


df_count_only = (

    df_orders

    .groupBy(
        "o_orderstatus"
    )

    .agg(
        F.count("*").alias(
            "order_count"
        )
    )
)


df_revenue_only = (

    df_orders

    .groupBy(
        "o_orderstatus")

    .agg(
        F.sum("o_totalprice").alias("total_revenue")))


print("\n--- Combined aggregation plan ---")

df_combined.explain("formatted")


print("\n--- Separate count aggregation plan ---")

df_count_only.explain("formatted")

print("\n--- Separate revenue aggregation plan ---")

df_revenue_only.explain("formatted")

print()

In [0]:
# ------------------------------------------------------------
# Step 14 — Performance Principle
# ------------------------------------------------------------

print(
    "=== STEP 14: PERFORMANCE PRINCIPLE ==="
)


print("""
When several measures:

- use the same source
- use the same grouping keys
- belong to the same target grain

calculate them together when practical.

Example:

groupBy("status").agg(
    count(...),
    sum(...),
    avg(...),
    min(...),
    max(...)
)

is generally preferable to independently executing several
equivalent groupBy queries just to calculate one measure each.


However:

Do NOT use these absolute statements:

"one groupBy = exactly one shuffle"

or

"one agg = exactly one physical pass"

Spark may use:

- partial aggregation
- multiple aggregate stages
- exchanges
- distinct aggregation stages
- Adaptive Query Execution
- optimizer rewrites

Inspect:

explain("formatted")

and Databricks Query Profile

when physical execution matters.
""")

In [0]:
# ------------------------------------------------------------
# Step 15 — Pivot Performance Awareness
# ------------------------------------------------------------

print(
    "=== STEP 15: PIVOT PERFORMANCE AWARENESS ==="
)


print("""
pivot() is useful when category values need to become columns.

Good example:

status:
F
O
P

→ only a few known output columns.


Risky example:

pivot(customer_id)

if there are hundreds of thousands of customers.

That could create an extremely wide schema and expensive processing.


Enterprise principle:

Use pivot() for bounded, meaningful category domains.

When possible, provide known pivot values explicitly:

pivot(
    "status",
    ["F", "O", "P"]
)
""")

In [0]:
# ------------------------------------------------------------
# Step 16 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
CONDITIONAL AGGREGATION:

sum(
    when(
        condition,
        measure
    ).otherwise(0)
)

→ Calculate a measure only for matching rows.


CONDITIONAL COUNT:

sum(
    when(
        condition,
        1
    ).otherwise(0)
)

→ Count rows satisfying a condition.


MULTIPLE MEASURES:

When measures share the same target grain,
calculate them together in one agg() when practical.


PIVOT-STYLE CONDITIONAL AGGREGATION:

groupBy(year).agg(
    revenue_F,
    revenue_O,
    revenue_P
)

→ Explicit and flexible.


ACTUAL pivot():

groupBy(year)
    .pivot(status, known_values)
    .agg(...)

→ Category values become columns.


PROGRAMMATIC AGGREGATIONS:

Build a Python list of Spark Column expressions.

Then:

agg(*expressions)

→ pass them into agg() dynamically.


RECONCILIATION:

Validate:

SUM(group counts)
=
source count

and when appropriate:

SUM(group measures)
=
global measure


PERFORMANCE:

groupBy() commonly requires data redistribution.

Related measures can often share grouping work.

Do NOT assume:

one groupBy
=
exactly one shuffle.


PLAN INSPECTION:

explain("formatted")
→ inspect logical/physical execution structure.

Query Profile
→ inspect actual Databricks execution behaviour.


CORE DESIGN QUESTION:

"What should one output row represent?"

Once the grain is known:

groupBy(keys)
     ↓
agg(measures)
     ↓
validate totals
     ↓
Gold-ready summary
""")


print("=== S3.16 CELL 2 COMPLETED ===")

# S3.16 — Key Takeaways: Multiple Aggregations

## 1. Conditional Aggregation

Conditional aggregation calculates a measure only for rows that satisfy a condition.

Example:

```python
F.sum(
    F.when(
        condition,
        measure
    )
)
```

Conceptually:

```text
Condition TRUE
→ include measure

Condition FALSE
→ NULL
→ sum() ignores NULL
```

### Without otherwise()

```python
F.sum(
    F.when(
        condition,
        measure
    )
)
```

If a group has no matching rows, the result can be:

```text
NULL
```

### With otherwise(0)

```python
F.sum(
    F.when(
        condition,
        measure
    ).otherwise(0)
)
```

If a group has no matching rows:

```text
result = 0
```

### Important

Choose between:

```text
NULL
```

and:

```text
0
```

based on business meaning.

Example:

```text
No fulfilled orders
→ fulfilled revenue = 0
```

may be appropriate.

---

# 2. Conditional Count

Two common patterns are:

### Pattern 1

```python
F.count(
    F.when(
        condition,
        1
    )
)
```

Non-matching rows produce NULL and `count(expression)` ignores NULL.

### Pattern 2

```python
F.sum(
    F.when(
        condition,
        1
    ).otherwise(0)
)
```

Conceptually:

```text
TRUE  → 1
FALSE → 0
```

Both patterns can be valid.

---

# 3. Why Use Conditional Aggregation Instead of filter()?

`filter()` removes non-matching rows.

Example:

```python
df.filter(
    F.col("status") == "F"
)
```

This is appropriate when the requirement is only about that subset.

Conditional aggregation is especially useful when several measures are
needed in the same grouped output.

Example:

```text
total revenue
fulfilled revenue
open revenue
fulfilled orders
open orders
```

can all be calculated at the same customer grain.

```text
Source rows
     ↓
groupBy(customer)
     ↓
Multiple conditional measures
     ↓
One customer summary row
```

---

# 4. Multiple Measures at the Same Grain

When related measures:

- use the same source
- use the same grouping keys
- belong to the same target grain

calculate them together in one `agg()` when practical.

Example:

```python
df.groupBy(
    "status"
).agg(
    F.count("*").alias("order_count"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order_value")
)
```

This allows Spark to share grouping work where possible.

Do not assume:

```text
one groupBy
=
exactly one shuffle
```

The physical execution plan depends on Spark optimisation and the
aggregate functions being used.

---

# 5. Pivot-Style Summary Using Conditional Aggregation

Example requirement:

```text
One row per year

Columns:
revenue_F
revenue_O
revenue_P
```

Conditional aggregation:

```python
df.groupBy(
    "order_year"
).agg(

    F.sum(
        F.when(
            F.col("status") == "F",
            F.col("amount")
        ).otherwise(0)
    ).alias("revenue_F"),

    F.sum(
        F.when(
            F.col("status") == "O",
            F.col("amount")
        ).otherwise(0)
    ).alias("revenue_O")
)
```

### Advantages

- Explicit
- Flexible
- Easy to apply different business conditions
- Direct control over output names

---

# 6. Actual pivot()

Spark also provides:

```python
df.groupBy(
    "order_year"
).pivot(
    "status",
    ["F", "O", "P"]
).agg(
    F.sum("amount")
)
```

Category values become columns.

Conceptually:

```text
status values:

F
O
P

        ↓ pivot()

columns:

F | O | P
```

### Known pivot values

When the valid domain is known, provide it explicitly:

```python
pivot(
    "status",
    ["F", "O", "P"]
)
```

This prevents Spark from first needing to discover all distinct pivot values
to determine the output columns.

It also makes the expected schema explicit.

---

# 7. Pivot Cardinality Warning

Pivot works best when the pivot column has a small, meaningful set of values.

Good example:

```text
status:
F
O
P
```

Risky example:

```text
customer_id:
hundreds of thousands of distinct values
```

A high-cardinality pivot can create:

```text
Very wide schema
      +
Expensive processing
```

Use pivot only when the category domain makes sense as columns.

---

# 8. Conditional Aggregation vs pivot()

| Requirement | Conditional aggregation | `pivot()` |
|---|---|---|
| Explicit business conditions | Excellent | Limited |
| Category values become columns | Yes, manually | Yes, automatically |
| Different logic per measure | Flexible | Less flexible |
| Small known category domain | Good | Good |
| Direct output naming | Easy | May require rename |
| High-cardinality category | Avoid | Especially avoid |

---

# 9. Programmatic Aggregations

When many similar measures are required, aggregation expressions can be
generated programmatically.

Example:

```python
statuses = [
    "F",
    "O",
    "P"
]

agg_exprs = [

    F.sum(
        F.when(
            F.col("status") == status,
            F.col("amount")
        ).otherwise(0)
    ).alias(
        f"revenue_{status}"
    )

    for status in statuses
]
```

Then:

```python
df.groupBy(
    "order_year"
).agg(
    *agg_exprs
)
```

---

# 10. What *agg_exprs Means

`agg_exprs` is a Python list containing Spark Column expressions.

Conceptually:

```text
[
    revenue_F expression,
    revenue_O expression,
    revenue_P expression
]
```

This:

```python
agg(
    *agg_exprs
)
```

uses Python argument unpacking.

It is equivalent conceptually to:

```python
agg(
    expression_1,
    expression_2,
    expression_3
)
```

The `*` is a Python feature, not a Spark-specific operator.

---

# 11. Reconciliation

Aggregation results should be validated.

## Count Reconciliation

If:

- no source rows were filtered out, and
- every source row belongs to exactly one grouping key,

then:

```text
SUM(group order counts)
=
source row count
```

Example:

```text
F orders
+
O orders
+
P orders
=
all source orders
```

---

## Conditional Count Reconciliation

Example:

```text
Source rows where status = F
=
SUM(fulfilled_orders across customer groups)
```

---

## Measure Reconciliation

Where the same source rows and NULL semantics are used:

```text
SUM(group revenue)
=
global revenue
```

Example:

```text
SUM(yearly revenue)
=
total source revenue
```

---

# 12. Performance

Related measures at the same grain can often share grouping work.

However, do not assume:

```text
one groupBy
=
one shuffle
```

or:

```text
one agg()
=
one physical pass
```

Spark may use:

```text
Partial aggregation
Exchange / shuffle
Multiple aggregation stages
Adaptive Query Execution
Optimizer rewrites
Distinct-aggregation stages
```

---

# 13. countDistinct() Performance

```python
F.countDistinct(
    "customer_id"
)
```

requires Spark to determine unique values across the required scope.

This can require more processing than:

```python
F.count("*")
```

Use `countDistinct()` when the business metric actually requires
distinct values.

---

# 14. Verify Execution

Use:

```python
df.explain(
    "formatted"
)
```

to inspect the physical-plan structure.

Use Databricks Query Profile when actual runtime execution details matter.

### Important

`explain()` helps you understand the plan.

It does not by itself prove that one implementation is faster.

Runtime performance depends on:

```text
Data volume
Data distribution
Cluster/serverless resources
Shuffle size
AQE
Caching/materialisation
Query plan
```

---

# 15. Quick Decision Guide

```text
Need one conditional measure?
→ sum(when(...))

Need conditional count?
→ count(when(...))
or
→ sum(when(..., 1).otherwise(0))

Need several measures at same grain?
→ combine in agg() when practical

Need categories as output columns?
→ conditional aggregation or pivot()

Need dynamic repetitive measures?
→ build expression list + agg(*exprs)

Need distinct entities?
→ countDistinct()

Need confidence totals are correct?
→ reconcile counts and measures

Need execution details?
→ explain("formatted") + Query Profile
```

---

# 16. Core Question

Before building any aggregation, ask:

> **What should ONE output row represent?**

Example:

```text
One row per customer?
→ groupBy(customer_id)

One row per year?
→ groupBy(order_year)

One row per year + status?
→ groupBy(order_year, status)
```

Once the grain is clear:

```text
Define grouping keys
      ↓
Build measures
      ↓
Use conditional logic where required
      ↓
Aggregate
      ↓
Reconcile
      ↓
Inspect execution when needed
      ↓
Gold-ready summary
```

**S3.16 — Multiple Aggregations completed.**

## Observed in our run (samples.tpch.orders)

- Top customer 721180: total 7,147,854.45 (38 orders)
  fulfilled 15 + open 20 = 35 orders, so 3 orders were missing
- The missing status was P (3 orders, revenue 436,340.19), verified by query
- Lesson: if status measures do not add up to the total, a status is missing
- In TPC-H, P = partially fulfilled (F = fulfilled, O = open)
- explain() on the status aggregation showed 3 shuffles:
  2 from count_distinct (group by status+customer, then by status)
  1 from orderBy (range partitioning)
- isFinalPlan=false, so AQE may change the runtime partition count
